In [ ]:
import fastf1
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
import json

YEAR = 2026

drivers_json_path = f'./data/{YEAR}/drivers_{YEAR}.json'
if not os.path.exists(drivers_json_path):
    raise FileNotFoundError(f"Drivers JSON file not found at {drivers_json_path}. Please run the driver data extraction first.")

with open(drivers_json_path, 'r') as f:
    drivers_list = json.load(f)
DRIVERS = ['_'.join(driver['id'].split('_')[:-1]) for driver in drivers_list]

schedule = fastf1.get_event_schedule(YEAR)
races = schedule["RoundNumber"][schedule["RoundNumber"] > 0].tolist()

# TODO:
# unprocessed_races = ...

unprocessed_races = [races[0]] # FIXME: temporary

for race in unprocessed_races:
    
    print(f"Processing race {race}...")
    
    session = fastf1.get_session(YEAR, race, "R")
    session.load(laps=True, telemetry=True, weather=False)
    lap_replays = [[] for _ in range(session.total_laps)]
    
    for idx, driver in enumerate(DRIVERS):
        driver_slug = f"{driver}_{YEAR}"
        for _, row in session.results.iterrows():
            if row["DriverId"] == driver:
                driver_abv = row["Abbreviation"]
                break
        driver_laps = session.laps.pick_drivers(driver_abv)
        if driver_laps.empty:
            continue
        
        print(f"\tProcessing driver [{idx+1}/{len(DRIVERS)}] {driver_slug}...........", end="")
        
        best_sectors = [float('inf'), float('inf'), float('inf')]
        best_lap_time = float('inf')
        
        for driver_lap in driver_laps.iterlaps():            
            lap_telemetry = driver_lap[1].get_telemetry()
            lap_number = int(driver_lap[1]['LapNumber'])
            
            best_sectors = [
                min(best_sectors[0], driver_lap[1]['Sector1Time'].total_seconds()),
                min(best_sectors[1], driver_lap[1]['Sector2Time'].total_seconds()),
                min(best_sectors[2], driver_lap[1]['Sector3Time'].total_seconds()),
            ]
            
            best_sectors = [0.0 if bs == float('inf') else bs for bs in best_sectors]
            
            best_lap_time = min(best_lap_time, driver_lap[1]['LapTime'].total_seconds())
            
            for _, row in lap_telemetry.iterrows():   
                lap_replays[lap_number - 1].append({
                    "driver": driver_slug,
                    "lap_number": lap_number,
                    "x": round(row['X'], 2),
                    "y": round(row['Y'], 2),
                    "z": round(row['Z'], 2),
                    "time": row['Time'].total_seconds(),
                    "position": driver_lap[1]['Position'], 
                    "compound": driver_lap[1]['Compound'],
                    "stint": driver_lap[1]['TyreLife'],
                    "gap_to_leader": 0, # TODO:
                    "gap_to_front": 0, # TODO:
                    "current_best_lap_time": best_lap_time,
                    "last_lap_time": driver_lap[1]['LapTime'].total_seconds(),
                    "current_sector_times": [
                        driver_lap[1]['Sector1Time'].total_seconds() if driver_lap[1]['Sector1Time'].total_seconds() > 0 else 0.0,
                        driver_lap[1]['Sector2Time'].total_seconds() if driver_lap[1]['Sector2Time'].total_seconds() > 0 else 0.0,
                        driver_lap[1]['Sector3Time'].total_seconds() if driver_lap[1]['Sector3Time'].total_seconds() > 0 else 0.0,
                    ],
                    "best_sector_time": best_sectors,
                    "is_in_pit": type(driver_lap[1]["PitInTime"].total_seconds()) is float, # TODO:
                    "is_retired": False, # TODO:
                })
        
        print(" [done]")
        
    os.makedirs(f'data/{YEAR}/replays/race_{race}', exist_ok=True)
    for idx, lap_replay in enumerate(lap_replays):
        lap_replay.sort(key=lambda x: x['time'])
        filename = f'data/{YEAR}/replays/race_{race}/replay_{YEAR}_race_{race}_lap_{idx+1}.json'
        with open(filename, 'w') as f:
            json.dump(lap_replay, f)


Processing race 1...
	Processing driver [1/30] albon_2026........... [done]
	Processing driver [2/30] alonso_2026........... [done]
	Processing driver [3/30] antonelli_2026........... [done]
	Processing driver [4/30] paul_aron_2026........... [done]
	Processing driver [5/30] bearman_2026........... [done]
	Processing driver [6/30] dino_beganovic_2026........... [done]
	Processing driver [7/30] bortoleto_2026........... [done]
	Processing driver [8/30] bottas_2026........... [done]
	Processing driver [9/30] luke_browning_2026........... [done]
	Processing driver [10/30] colapinto_2026........... [done]
	Processing driver [11/30] jak_crawford_2026........... [done]
	Processing driver [12/30] leonardo_fornaroli_2026........... [done]
	Processing driver [13/30] gasly_2026........... [done]
	Processing driver [14/30] hadjar_2026........... [done]
	Processing driver [15/30] hamilton_2026........... [done]
	Processing driver [16/30] colton_herta_2026........... [done]
	Processing driver [17/3

In [ ]:
import fastf1
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
import json
import pandas as pd

YEAR = 2026

drivers_json_path = f'./data/{YEAR}/drivers_{YEAR}.json'
if not os.path.exists(drivers_json_path):
    raise FileNotFoundError(f"Drivers JSON file not found at {drivers_json_path}. Please run the driver data extraction first.")

with open(drivers_json_path, 'r') as f:
    drivers_list = json.load(f)
DRIVERS = ['_'.join(driver['id'].split('_')[:-1]) for driver in drivers_list]

schedule = fastf1.get_event_schedule(YEAR)
races = schedule["RoundNumber"][schedule["RoundNumber"] > 0].tolist()

# TODO:
# unprocessed_races = ...

unprocessed_races = [races[0]] # FIXME: temporary

COLUMNS = [
    "driver", "lap_number", "x", "y", "z", "time", "position", "compound", "stint",
    "gap_to_leader", "gap_to_front", "current_best_lap_time", "last_lap_time",
    "current_sector1_time", "current_sector2_time", "current_sector3_time",
    "best_sector1_time", "best_sector2_time", "best_sector3_time",
    "is_in_pit", "is_retired",
]

DTYPES = {
    "driver": "category",
    "lap_number": "uint16",
    "x": "float32",
    "y": "float32",
    "z": "float32",
    "time": "float32",
    "position": "uint16",
    "compound": "category",
    "stint": "uint16",
    "gap_to_leader": "float32",
    "gap_to_front": "float32",
    "current_best_lap_time": "float32",
    "last_lap_time": "float32",
    "current_sector1_time": "float32",
    "current_sector2_time": "float32",
    "current_sector3_time": "float32",
    "best_sector1_time": "float32",
    "best_sector2_time": "float32",
    "best_sector3_time": "float32",
    "is_in_pit": "bool",
    "is_retired": "bool",
}

for race in unprocessed_races:
    
    print(f"Processing race {race}...")
    
    session = fastf1.get_session(YEAR, race, "R")
    session.load(laps=True, telemetry=True, weather=False)
    lap_replays = [[] for _ in range(session.total_laps)]
    
    for idx, driver in enumerate(DRIVERS):
        driver_slug = f"{driver}_{YEAR}"
        for _, row in session.results.iterrows():
            if row["DriverId"] == driver:
                driver_abv = row["Abbreviation"]
                break
        driver_laps = session.laps.pick_drivers(driver_abv)
        if driver_laps.empty:
            continue
        
        print(f"\tProcessing driver [{idx+1}/{len(DRIVERS)}] {driver_slug}{'.'*(30-len(driver_slug))}", end="")
        
        best_sectors = [float('inf'), float('inf'), float('inf')]
        best_lap_time = float('inf')
        session_start_time = session.session_start_time.total_seconds()
        
        for driver_lap in driver_laps.iterlaps():            
            lap_telemetry = driver_lap[1].get_telemetry()
            lap_number = int(driver_lap[1]['LapNumber'])
            
            sector1_time = driver_lap[1]['Sector1Time'].total_seconds()
            sector2_time = driver_lap[1]['Sector2Time'].total_seconds()
            sector3_time = driver_lap[1]['Sector3Time'].total_seconds()
            
            lap_time = driver_lap[1]['LapTime'].total_seconds()
            
            for _, row in lap_telemetry.iterrows():  
                
                curr_time_lap = row['Time'].total_seconds()
                
                s1 = min(curr_time_lap, sector1_time) 
                s2 = min(curr_time_lap - s1, sector2_time)
                s3 = min(curr_time_lap - s1 - s2, sector3_time)
                
                lap_replays[lap_number - 1].append([
                    driver_slug,                # driver
                    lap_number,                 # lap_number
                    round(row['X'], 2),         # x
                    round(row['Y'], 2),         # y
                    round(row['Z'], 2),         # z
                    row['SessionTime'].total_seconds() - session_start_time,# time
                    int(driver_lap[1]['Position']) if not pd.isna(driver_lap[1]['Position']) else len(DRIVERS),  # position
                    driver_lap[1]['Compound'],  # compound
                    int(driver_lap[1]['TyreLife']),  # stint
                    0, # TODO:                  # gap_to_leader
                    0, # TODO:                  # gap_to_front
                    best_lap_time if best_lap_time != float('inf') else 0.0,  # current_best_lap_time
                    lap_time if lap_number > 1 else 0.0, # last_lap_time
                    s1 if s1 > 0 else 0.0,      # current_sector1_time
                    s2 if s2 > 0 else 0.0,      # current_sector2_time
                    s3 if s3 > 0 else 0.0,      # current_sector3_time
                    best_sectors[0] if best_sectors[0] != float('inf') else 0.0,    # best_sector1_time
                    best_sectors[1] if best_sectors[1] != float('inf') else 0.0,    # best_sector2_time
                    best_sectors[2] if best_sectors[2] != float('inf') else 0.0,    # best_sector3_time
                    driver_lap[1]["PitInTime"] is not pd.NaT, # is_in_pit
                    False,                      # TODO: # is_retired
                ])
            
            best_sectors = [
                min(best_sectors[0], sector1_time),
                min(best_sectors[1], sector2_time),
                min(best_sectors[2], sector3_time),
            ]
            
            best_lap_time = min(best_lap_time, lap_time)
        
        print("[done]")
        
    os.makedirs(f'data/{YEAR}/replays', exist_ok=True)
    
    all_rows = []
    for lap_replay in lap_replays:
        lap_replay.sort(key=lambda x: x[5]) # based on time
        all_rows.extend(lap_replay)
    
    df = pd.DataFrame(all_rows, columns=COLUMNS).astype(DTYPES)
    filename = f'data/{YEAR}/replays/replay_{YEAR}_race_{race}.parquet'
    df.to_parquet(filename, engine='fastparquet', compression='zstd', index=False)
    print(f"Wrote {len(df):,} rows to {filename} ({os.path.getsize(filename)/1_048_576:.2f} MiB)")
    
    filename = f'data/{YEAR}/replays/replay_{YEAR}_race_{race}.csv'
    df.to_csv(filename, index=False)
    print(f"Wrote {len(df):,} rows to {filename} ({os.path.getsize(filename)/1_048_576:.2f} MiB)")

Processing race 1...
	Processing driver [1/22] norris_2026...................[done]
	Processing driver [2/22] max_verstappen_2026...........[done]
	Processing driver [3/22] antonelli_2026................[done]
	Processing driver [4/22] leclerc_2026..................[done]
	Processing driver [5/22] hamilton_2026.................[done]
	Processing driver [6/22] hadjar_2026...................[done]
	Processing driver [7/22] russell_2026..................[done]
	Processing driver [8/22] lawson_2026...................[done]
	Processing driver [10/22] arvid_lindblad_2026...........[done]
	Processing driver [11/22] bortoleto_2026................[done]
	Processing driver [12/22] gasly_2026....................[done]
	Processing driver [13/22] stroll_2026...................[done]
	Processing driver [14/22] alonso_2026...................[done]
	Processing driver [15/22] colapinto_2026................[done]
	Processing driver [16/22] ocon_2026.....................[done]
	Processing driver [17/22] 

In [47]:
import fastf1
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
import pandas as pd
session = fastf1.get_session(2026, 1, "R")
session.load(laps=True, telemetry=True, weather=False)
driver_laps = session.laps.pick_drivers("VER")
driver_laps['PitInTime']
session_start_time = session.session_start_time.total_seconds()
for driver_lap in driver_laps.iterlaps():            
    lap_telemetry = driver_lap[1].get_telemetry()
    # print(lap_telemetry['DriverAhead'])
    # print(driver_lap[1]['PitInTime'] is pd.NaT)
    print(driver_lap[1]['Sector1Time'], driver_lap[1]['Sector2Time'], driver_lap[1]['Sector3Time'])
    # for _, row in lap_telemetry.iterrows():
        # print(row['SessionTime'].total_seconds() - session_start_time)
        # print(row['Time'].total_seconds())
    

NaT 0 days 00:00:18.975000 0 days 00:00:39.753000
0 days 00:00:31.200000 0 days 00:00:18.005000 0 days 00:00:37.623000
0 days 00:00:31.213000 0 days 00:00:18.031000 0 days 00:00:37.852000
0 days 00:00:30.547000 0 days 00:00:17.928000 0 days 00:00:38.096000
0 days 00:00:29.902000 0 days 00:00:18.360000 0 days 00:00:36.617000
0 days 00:00:30.063000 0 days 00:00:17.806000 0 days 00:00:37.099000
0 days 00:00:30.335000 0 days 00:00:18.224000 0 days 00:00:36.825000
0 days 00:00:29.790000 0 days 00:00:18.051000 0 days 00:00:36.866000
0 days 00:00:29.879000 0 days 00:00:17.966000 0 days 00:00:37.226000
0 days 00:00:30.036000 0 days 00:00:17.609000 0 days 00:00:37.488000
0 days 00:00:29.804000 0 days 00:00:18.006000 0 days 00:00:43.219000
0 days 00:00:39.818000 0 days 00:00:27.803000 0 days 00:00:46.593000
0 days 00:00:40.500000 0 days 00:00:28.662000 0 days 00:00:45.699000
0 days 00:00:31.326000 0 days 00:00:18.104000 0 days 00:00:36.700000
0 days 00:00:29.618000 0 days 00:00:17.939000 0 days 

KeyboardInterrupt: 